In [7]:
import pandas as pd

df_claim = pd.read_csv('final-dataset(A1-31306samples)-train-topicmodel.csv')
df_claim.head()

,publication_number,country_code,kind_code,title,abstract,claims,publication_date,ipc_code,cpc_code,first_claim,claim_lengths,sub_classes,sub_class,abstract_lengths
0,US2020097067A1,US,A1,Artificial Intelligence System and Interactive...,"A reality interactive responding system, compr...","1 . A reality interactive responding system, c...",20200326,G06F9/448,G16H40/67,"1 . A reality interactive responding system, c...",200,"['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",123
1,US2020098473A1,US,A1,Data Storage and Retrieval System for Non-Cont...,A web-based interface enables medical personne...,What is claimed is: \n \n 1 . A da...,20200326,G16H40/67,H04L67/1097,What is claimed is: 1 . A data storage and ret...,200,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",142
2,US2020098451A1,US,A1,Hybrid analysis framework for prediction of ou...,A facility for predicting patient outcomes on ...,"1 . A method in a computing system, comprising...",20200326,G16H10/20,G16H10/20,"1 . A method in a computing system, comprising...",200,"['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",81
3,US2020098458A1,US,A1,Medical cannabis platform with physician and p...,"Through a physician&#39;s portal, a platform c...",What is claimed is: \n \n 1 . A me...,20200326,G16H80/00,A61K36/185,What is claimed is: 1 . A method for providing...,200,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",148
4,US2020093988A1,US,A1,Patient day planning systems and methods,"Infusion systems, infusion devices, and relate...",What is claimed is: \n \n 1 . A me...,20200326,G16H20/17,A61M2230/201,What is claimed is: 1 . A method of monitoring...,200,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",110


In [8]:
len(df_claim)

31306

In [9]:
first_record = df_claim['first_claim'][1]
first_record

'What is claimed is: 1 . A data storage and retrieval system for non-contiguous medical device data, the system comprising: a medical device, connectable to a computer network, subject to occasional gaps in connectivity to the computer network, and configured to automatically repeatedly capture status information about the medical device and send messages containing the status information via the computer network; a network connectivity log configured to automatically record: (a) times at which the medical device connects to the computer network and (b) times at which the medical device disconnects from the computer network; a data store configured to automatically: store digital media data in a media file; and provide a requested portion, less than all, of the stored media file in response to a provision request, wherein the provision request includes an index, relative to an end of the media file, that corresponds to the requested portion; a media server, connectable to the computer 

In [10]:
x = len(df_claim['first_claim'][1])
print(x)

1331


In [11]:
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

# Check if GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Sentence Transformers model
sentence_model = SentenceTransformer('AI-Growth-Lab/PatentSBERTa')
#sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Adjust UMAP hyperparameters
umap_model = UMAP(n_neighbors=3, 
                  n_components=3, 
                  min_dist=0.05, 
                  metric='cosine', 
                  random_state=100,
                  n_jobs=-1)  # Utilize all available CPU cores for UMAP preprocessing

# Adjust HDBSCAN hyperparameters
hdbscan_model = HDBSCAN( min_cluster_size=80,
                        min_samples=40, 
                        metric='euclidean', 
                        cluster_selection_method='eom', 
                        prediction_data=True)

# Adjust CountVectorizer hyperparameters
vectorizer_model = CountVectorizer(ngram_range=(1, 3), min_df=10, max_df=0.5)


# c-TF-IDF
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# Run Model
topic_model = BERTopic(umap_model=umap_model,
                       embedding_model=sentence_model, 
                       hdbscan_model=hdbscan_model, 
                       vectorizer_model=vectorizer_model,
                       ctfidf_model=ctfidf_model,
                       verbose=True)

# Replace 'df['first_claim']' with your data source for the first_claim column
topics, probabilities = topic_model.fit_transform(df_claim['first_claim'])


Batches:   0%|          | 0/979 [00:00<?, ?it/s]

2023-09-24 11:30:47,513 - BERTopic - Transformed documents to Embeddings
2023-09-24 11:31:34,730 - BERTopic - Reduced dimensionality
2023-09-24 11:31:37,339 - BERTopic - Clustered reduced embeddings


In [12]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

documents = pd.DataFrame({"Document": df_claim['first_claim'],
                          "ID": range(len(df_claim['first_claim'])),
                          "Topic": topics})

documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)

# Extract vectorizer and analyzer from BERTopic
vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

# Extract features for Topic Coherence evaluation
words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] for topic in range(len(set(topics))-1)]


# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_npmi')
coherence_c_nmpi = coherence_model.get_coherence()
print("c_npmi is: ",coherence_c_nmpi)

c_npmi is:  -0.15626478182216264


In [13]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

documents = pd.DataFrame({"Document":df_claim['first_claim'],
                          "ID": range(len(df_claim['first_claim'])),
                          "Topic": topics})
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)

# Extract vectorizer and analyzer from BERTopic
vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

# Extract features for Topic Coherence evaluation
words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_v')
coherence_CV= coherence_model.get_coherence()
print("C_V is: ", coherence_CV)

C_V is:  0.32861081695473926


In [14]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

documents = pd.DataFrame({"Document":df_claim['first_claim'],
                          "ID": range(len(df_claim['first_claim'])),
                          "Topic": topics})
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)

# Extract vectorizer and analyzer from BERTopic
vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

# Extract features for Topic Coherence evaluation
words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

# Evaluate
coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='u_mass')
coherence_u_mass= coherence_model.get_coherence()
print("u_mass is: ", coherence_u_mass)

u_mass is:  -0.5913988250513408


In [15]:
 topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,12235,-1_sleep_clinical trial_ecg_the surgical,"[sleep, clinical trial, ecg, the surgical, med...",[1 ) SYSTEM FOR THE PREVENTION AND PREDICTION ...
1,0,5409,0_claimed is system_is system_is system for_cl...,"[claimed is system, is system, is system for, ...","[What is claimed is: 1 . A system for a user, ..."
2,1,1388,1_health care provider_of claim method_medical...,"[health care provider, of claim method, medica...",[1 . A computer-implemented method for providi...
3,2,1208,2_claimed is computer_is method comprising_the...,"[claimed is computer, is method comprising, th...",[What is claimed is: 1 . A computerized method...
4,3,794,3_medical images_the medical image_image the m...,"[medical images, the medical image, image the ...",[What is claimed is: 1 . A method for converti...
5,4,751,4_medical images_medical imaging_exam_in claim...,"[medical images, medical imaging, exam, in cla...",[1 . A method of administering medical digital...
6,5,693,5_personal health_care system_data the system_...,"[personal health, care system, data the system...",[1 . An information management system comprisi...
7,6,467,6_claim 21_20 canceled_canceled 21_20 canceled 21,"[claim 21, 20 canceled, canceled 21, 20 cancel...",[1 - 11 . (canceled) 12 - 18 . (canceled) 19 ....
8,7,465,7_gene_dna_genes_nucleic,"[gene, dna, genes, nucleic, genomic, sequencin...",[What is claimed is: 1 . A method for determin...
9,8,404,8_insulin_blood glucose_glucose level_bolus,"[insulin, blood glucose, glucose level, bolus,...",[1 . A method of regulating blood glucose usin...


In [16]:
#adding topics and probs for eachdoc in dataset
df_claim['topics'] = topics
df_claim['prob'] = probabilities
df_claim_topic=df_claim[["publication_number","title","first_claim","sub_classes","sub_class","topics","prob"]]
df_claim_topic

,publication_number,title,first_claim,sub_classes,sub_class,topics,prob
0,US2020097067A1,Artificial Intelligence System and Interactive...,"1 . A reality interactive responding system, c...","['G06F9/448', 'G16H40/67']","['G06F', 'G16H']",-1,0.000000
1,US2020098473A1,Data Storage and Retrieval System for Non-Cont...,What is claimed is: 1 . A data storage and ret...,"['G16H40/67', 'H04L67/1097']","['G16H', 'H04L']",0,1.000000
2,US2020098451A1,Hybrid analysis framework for prediction of ou...,"1 . A method in a computing system, comprising...","['G16H10/20', 'G16H10/20']","['G16H', 'G16H']",-1,0.000000
3,US2020098458A1,Medical cannabis platform with physician and p...,What is claimed is: 1 . A method for providing...,"['G16H80/00', 'A61K36/185']","['G16H', 'A61K']",-1,0.000000
4,US2020093988A1,Patient day planning systems and methods,What is claimed is: 1 . A method of monitoring...,"['G16H20/17', 'A61M2230/201']","['G16H', 'A61M']",47,1.000000
...,...,...,...,...,...,...,...
31301,US2016253489A1,User authentication system,1 . A user authentication system comprising: a...,"['G16H10/60', 'G06F21/32']","['G16H', 'G06F']",5,0.900515
31302,US2016253467A1,"Diagnosis support apparatus and method, and no...",What is claimed is: 1 . A diagnosis support ap...,"['G16H10/60', 'A61B5/743']","['G16H', 'A61B']",0,1.000000
31303,US2016253462A1,Novel open-access scheduling system that optim...,1 . A medical appointment scheduling system co...,"['G16H40/20', 'G06F19/327']","['G16H', 'G06F']",-1,0.000000
31304,US2016249985A1,Interrelated point acquisition for navigated s...,"1 . A data processing, comprising a computer h...","['G16H20/40', 'G06F19/324']","['G16H', 'G06F']",4,0.996585


# prediction

In [17]:
import pandas as pd

df_claim_test = pd.read_csv('test-queries-USPTO(A1)-2023-G16H.csv')
df_claim_test.head()

,publication_numbers,abstract,first_claim,class_codes
0,US20230238130A1,A physiological sensor has light emitting sour...,1. A physiological monitoring device comprisin...,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/..."
1,US20230270344A1,A wearable monitoring device includes a band c...,"1. A monitoring device, comprising:\na band co...","A61B5/02405,A61B5/01,A61B5/16,A61B5/02055,A61B..."
2,US20230200909A1,A number of improvements are provided relating...,1-25. (canceled) 26. A method for guiding a fr...,"A61B17/17,A61B2090/061,A61B90/06,A61B2090/365,..."
3,US20230218347A1,Embodiments include a system for determining c...,1-184. (canceled) 185. A computer-implemented ...,"G06V10/46,G06V20/698,G06T2207/20112,G06T7/13,A..."
4,US20230063013A1,A community based response system for providin...,1. (canceled) 2. A community based response sy...,"H04M1/72418,G08B,G08,G08B25/016,G,H04W4/023,G1..."


# test1

In [18]:
df_claim_test.iloc[0]


publication_numbers                                      US20230238130A1
abstract               A physiological sensor has light emitting sour...
first_claim            1. A physiological monitoring device comprisin...
class_codes            A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...
Name: 0, dtype: object

In [19]:
test1 = df_claim_test.loc[0, 'first_claim']
test1

'1. A physiological monitoring device comprising:\nat least two LEDs, the at least two LEDs configured to emit light of at least two different wavelengths; at least one detector configured to detect at least a portion of the light emitted from the at least two LEDs after at least a portion of the light has been attenuated by tissue, the at least one detector configured to output at least one signal responsive to the detected light; a light block surrounding the at least one detector, the light block forming a cavity, the light block comprising a light-absorbing material, the light block including only one circular opening through which light is configured to pass, an area of the circular opening being smaller than a surface area of a facing surface of the at least one detector; and a processor configured to receive and process one or more signals responsive to the outputted at least one signal and determine a physiological parameter of a user responsive to the one or more signals. at l

In [20]:
import numpy as np 

# Find topics
num_of_topics = 5
similar_topics, similarity = topic_model.find_topics(test1, top_n=num_of_topics); 

# Print results
print(f'The top {num_of_topics} similar topics are {similar_topics}, and the similarities are {np.round(similarity,2)}')

for idx, topic_idx in enumerate(similar_topics):
    topic = topic_model.get_topic(topic_idx)
    keywords = ' '.join(str(keyword) for keyword in topic[0])
    print(f"Topic {idx+1}: {keywords}")

The top 5 similar topics are [16, 25, 9, -1, 39], and the similarities are [0.79 0.79 0.79 0.78 0.77]
Topic 1: medical device of 0.27979274135764304
Topic 2: physiological signal 0.2792462222724727
Topic 3: the apparatus of 0.454426918737517
Topic 4: sleep 0.1240670651751376
Topic 5: nurse 0.33454800955968295


In [21]:
filter_topics_filter = df_claim_topic[df_claim_topic['topics'] == 16]
filter_topics_filter = filter_topics_filter.sort_values('prob', ascending=False)
filter_topics_filter

,publication_number,title,first_claim,sub_classes,sub_class,topics,prob
16081,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.000000
10046,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.000000
13619,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.000000
13552,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.000000
13542,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.000000
...,...,...,...,...,...,...,...
12874,US2021307622A1,Doctor health monitoring system based on medic...,1 : A doctor health monitoring system based on...,"['A61B5/024', 'G16H50/20']","['A61B', 'G16H']",16,0.704687
10466,US2017325700A1,Real-time vagal monitoring and intervention,1 . (canceled) 2 . The monitoring device of cl...,"['G16H40/63', 'A61B5/02405']","['G16H', 'A61B']",16,0.703344
25894,US2011106553A1,Health management guideline advising device,1 . A health management guideline advising dev...,"['G16H20/30', 'G09B19/0092']","['G16H', 'G09B']",16,0.702570
25850,US2011093210A1,Measurement device and method of controlling t...,1 . A measurement device comprising: a first m...,"['G16H40/67', 'G16H40/67']","['G16H', 'G16H']",16,0.701239


In [22]:
# Define the range of topic IDs you want to retrieve documents for
selected_topic_range =[16, 25, 9, -1, 39]  # Replace with your desired range of topic IDs

# Create an empty DataFrame to store the top 10 documents
top_10_documents_test1 = pd.DataFrame()

# Iterate through the selected topic IDs
for selected_topic_id in selected_topic_range:
    # Filter the dataset based on the current topic ID
    filtered_df = df_claim_topic[df_claim_topic['topics'] == selected_topic_id]
    
    # Sort the filtered dataset by similarity scores in descending order
    sorted_df = filtered_df.sort_values(by='prob', ascending=False)
    
    # Retrieve the top 10 documents for the current topic
    top_10_for_topic = sorted_df.head(10)
    
    # Append the top 10 documents for the current topic to the result DataFrame
    top_10_documents_test1 = top_10_documents_test1.append(top_10_for_topic)

# Display the top 10 documents for all selected topics
top_10_documents_test1

,publication_number,title,first_claim,sub_classes,sub_class,topics,prob
16081,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0
10046,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0
13619,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0
13552,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0
13542,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0
13476,US2012265556A1,Method and device for remote controlled applic...,55 . A device useful for monitoring of patient...,"['G16H40/67', 'A61B5/0022']","['G16H', 'A61B']",16,1.0
13366,US2012239705A1,Mobile information system for 12-lead ecg,1 . A 12-leads electrocardiograph mobile infor...,"['G06F17/30', 'G16H10/60']","['G06F', 'G16H']",16,1.0
23224,US2010217803A1,Interface device for communication between a m...,1 . A wireless interface device for communicat...,"['G16H40/67', 'G16H40/67']","['G16H', 'G16H']",16,1.0
23362,US2010253509A1,Personal environmental monitoring method and s...,"1 . A portable personal environmental monitor,...","['G08B1/08', 'G16H50/30']","['G08B', 'G16H']",16,1.0
23479,US2010283601A1,Medication usage monitoring and reminding devi...,"1 ) A device for monitoring medication usage, ...","['G16H10/60', 'A61J7/0418']","['G16H', 'A61J']",16,1.0


In [23]:
num_records = top_10_documents_test1.shape[0]
num_records

50

# list of 100 queries

In [25]:
# Assuming you want to predict topics for the first 100 samples in 'df_claim_test'
num_samples_to_predict = 100

num_of_topics = 5
results = []

# Assuming you have a list of 100 queries in test
for query in df_claim_test['first_claim'][:num_samples_to_predict]:
    if isinstance(query, str):  # Check if the query is a string
        similar_topics, similarity = topic_model.find_topics(query, top_n=num_of_topics)
        results.append((similar_topics, similarity))
    else:
        # Handle cases where 'query' is not a string (e.g., it's a float)
        print(f"Skipping non-string query: {query}")

# Now, the 'results' list contains the similar topics and similarities for each query
# You can access the results for a specific query like this:
for i, (similar_topics, similarity) in enumerate(results):
    print(f"Query {i + 1}: Similar Topics {similar_topics}, Similarity {similarity}")


Skipping non-string query: nan
Skipping non-string query: nan
Query 1: Similar Topics [16, 25, 9, -1, 39], Similarity [0.7918744, 0.79018235, 0.7878056, 0.778339, 0.76860034]
Query 2: Similar Topics [16, 25, 22, 39, 40], Similarity [0.7235428, 0.71160823, 0.69949853, 0.68764395, 0.68443054]
Query 3: Similar Topics [6, 42, 19, 15, 21], Similarity [0.7082089, 0.69748014, 0.6785549, 0.6728228, 0.6512108]
Query 4: Similar Topics [19, 6, 34, 38, 17], Similarity [0.68264574, 0.66813165, 0.649974, 0.64888793, 0.6253784]
Query 5: Similar Topics [42, 24, 19, -1, 6], Similarity [0.68696046, 0.6848376, 0.6762146, 0.6728812, 0.6684448]
Query 6: Similar Topics [20, 25, 17, 12, 16], Similarity [0.7796078, 0.7527888, 0.7323582, 0.72718954, 0.72086835]
Query 7: Similar Topics [22, 40, 24, 16, 33], Similarity [0.83270305, 0.80476594, 0.7996713, 0.7931266, 0.79160213]
Query 8: Similar Topics [9, 28, 40, 24, 16], Similarity [0.7410316, 0.7391194, 0.7045108, 0.7033888, 0.69814295]
Query 9: Similar Topics 

In [28]:
# Assuming you want to predict topics for the first 100 samples in 'df_Abstract_test'
num_samples_to_predict = 100

num_of_topics = 5
results = []

# Create an empty DataFrame to store the results
result_df_q = pd.DataFrame(columns=['query_publication_numbers', 'query_class_codes', 'query_claim', 'query_predicted_topics'])

# Assuming you have a list of 100 queries in 'df_Abstract_test'
for i, (query, publication_number, first_claim, class_codes) in enumerate(zip(df_claim_test['first_claim'][:num_samples_to_predict], 
                                                    df_claim_test['publication_numbers'][:num_samples_to_predict],
                                                    df_claim_test['first_claim'][:num_samples_to_predict],
                                                    df_claim_test['class_codes'][:num_samples_to_predict])):
    # Check if the 'first_claim' is a string before processing
    if isinstance(query, str):
        similar_topics, similarity = topic_model.find_topics(query, top_n=num_of_topics)
        results.append((similar_topics, similarity))
        
        # Store the results in the DataFrame
        result_df_q = result_df_q.append({
            'query_claim': query,
            'query_predicted_topics': similar_topics,
            'query_publication_numbers': publication_number,
            'query_class_codes': class_codes
        }, ignore_index=True)
    else:
        # Handle cases where 'query' is not a string (e.g., it's a nan value)
        print(f"Skipping non-string query: {query}")

result_df_q


Skipping non-string query: nan
Skipping non-string query: nan


,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics
0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,"[16, 25, 9, -1, 39]"
1,US20230270344A1,"A61B5/02405,A61B5/01,A61B5/16,A61B5/02055,A61B...","1. A monitoring device, comprising:\na band co...","[16, 25, 22, 39, 40]"
2,US20230200909A1,"A61B17/17,A61B2090/061,A61B90/06,A61B2090/365,...",1-25. (canceled) 26. A method for guiding a fr...,"[6, 42, 19, 15, 21]"
3,US20230218347A1,"G06V10/46,G06V20/698,G06T2207/20112,G06T7/13,A...",1-184. (canceled) 185. A computer-implemented ...,"[19, 6, 34, 38, 17]"
4,US20230063013A1,"H04M1/72418,G08B,G08,G08B25/016,G,H04W4/023,G1...",1. (canceled) 2. A community based response sy...,"[42, 24, 19, -1, 6]"
...,...,...,...,...
93,US20230017310A1,"G06Q10/10,G,G06F,G06F16/90,G06F16/95,G16H30/20...",1. A system for storing medical information re...,"[24, 13, 22, 32, 16]"
94,US20230091925A1,"H04L41/5061,G,G06,G06F,H04L67/00,H,H04,G06F16/...",1. A method for event notification in an inter...,"[33, 12, 25, 40, -1]"
95,US20230010638A1,"A61M5/142,A61,Y10S128/00,G06T2219/2016,G06F8/6...",1.-2. (canceled) 3. A method of managing an in...,"[38, 19, 6, 42, 24]"
96,US20230009812A1,"A61B5/14551,A61,G,A61B5/021,A61B5/746,A61B,A61...",1. (canceled) 2. A patient monitoring system c...,"[42, 32, 19, -1, 6]"


In [29]:
import pandas as pd

# Create an empty DataFrame to store the results
result_df = pd.DataFrame(columns=['publication_number', 'title', 'first_claim', 'sub_classes', 'sub_class', 'topics', 'prob', 'query_publication_numbers', 'query_class_codes', 'query_claim', 'query_predicted_topics'])

# Define the number of documents to retrieve for each topic
num_of_documents_to_retrieve = 10

for i, row in result_df_q.iterrows():
    query = row['query_claim']
    predicted_topics = row['query_predicted_topics']
    
    for topic_id in predicted_topics:
        # Filter 'df_Abstract_topic' to get the top 'num_of_documents_to_retrieve' documents for the current topic_id
        topic_documents = df_claim_topic[df_claim_topic['topics'] == topic_id]
        
        # Sort the documents by probability in descending order
        topic_documents = topic_documents.sort_values(by='prob', ascending=False).head(num_of_documents_to_retrieve)
        
        # Append the results to the 'result_df' DataFrame
        for _, doc_row in topic_documents.iterrows():
            result_df = result_df.append({
                'query_publication_numbers': row['query_publication_numbers'],
                'query_class_codes': row['query_class_codes'],
                'query_claim': query,
                'query_predicted_topics': [topic_id],  # Assign the current topic_id as a list
                'publication_number': doc_row['publication_number'],
                'title': doc_row['title'],
                'first_claim': doc_row['first_claim'],
                'sub_classes': doc_row['sub_classes'],
                'sub_class': doc_row['sub_class'],
                'topics': doc_row['topics'],
                'prob': doc_row['prob']
            }, ignore_index=True)

# Now, 'result_df' contains the top 10 most probable documents for each predicted topic list for each query
result_df


,publication_number,title,first_claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics
0,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16]
1,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16]
2,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16]
3,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16]
4,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16]
...,...,...,...,...,...,...,...,...,...,...,...
4895,US2018068076A1,Systems and methods for semantic search and ex...,1 . A system that facilitates using user-enter...,"['G16H50/20', 'G06F16/90344']","['G16H', 'G06F']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32]
4896,US2007055543A1,Medical resource estimation and simulation system,1 . A healthcare decision support system for s...,"['G16H10/60', 'G16Z99/00']","['G16H', 'G16Z']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32]
4897,US2007055545A1,System and user interface for processing patie...,1 . A system for use in processing patient cli...,"['G06F19/00', 'G16H50/20']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32]
4898,US2006271556A1,System and method for integration of medical i...,1 . A system for integrating medical informati...,"['G06F17/30', 'G16H70/00']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32]


In [30]:
result_df['query_class_codes'][0]

'A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/7221,A61B5/14532,G16H40/00,A61B5/14546,G16H40/60,A61B5/68,A61B2562/08,H,A61B1/00,A61B2562/221,A61B5/7475,A61B2562/18,A61B5/746,A61B5/7405,H05K,G16H40/67,A61B,Y10,A61B5/6813,Y,A61B2562/185,G16H10/40,A61B5/6826,A61B5/02,A61B5/0002,A61B5/1455,A61B5/6832,A,A61B5/7235,A61B5/6801,A61B5/7278,G16H,A61B5/026,A61B5/742,A61B2562/22,A61B5/024,G16,A61B5/02416,A61B5/683,A61B5/0015,A61B5/6825,Y10S,A61B5/1495,A61B5/72,A61B5/0261,A61B2562/085,A61B2562/222,A61B5/7275,A61B5/00,A61B5/145,A61B5/74,G16H10/00,H05K999/99,A61B5/7246,Y10S439/909,A61B5/14552,A61B5/6838,A61B5/6815,A61B5/7271,H05K999/00,A61B5/02427,A61B5/0205,Y10S439/00,A61B5/0022,A61B5/6814,A61B2562/00,H05'

In [31]:
# Add a new column to store the filtered codes
result_df['query_codes_G16H'] = ''

# Define a function to extract codes starting with 'G61H' from the class codes
def extract_G16H_codes(class_codes):
    codes = class_codes.split(',')
    return ','.join([code for code in codes if code.startswith('G16H')])

# Iterate through rows and update the 'query_codes_G61H' column
for index, row in result_df.iterrows():
    class_codes = row['query_class_codes']
    filtered_codes = extract_G16H_codes(class_codes)
    result_df.at[index, 'query_codes_G16H'] = filtered_codes

# Display the updated DataFrame
result_df


,publication_number,title,first_claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H
0,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
1,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
2,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
3,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
4,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G..."
...,...,...,...,...,...,...,...,...,...,...,...,...
4895,US2018068076A1,Systems and methods for semantic search and ex...,1 . A system that facilitates using user-enter...,"['G16H50/20', 'G06F16/90344']","['G16H', 'G06F']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60"
4896,US2007055543A1,Medical resource estimation and simulation system,1 . A healthcare decision support system for s...,"['G16H10/60', 'G16Z99/00']","['G16H', 'G16Z']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60"
4897,US2007055545A1,System and user interface for processing patie...,1 . A system for use in processing patient cli...,"['G06F19/00', 'G16H50/20']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60"
4898,US2006271556A1,System and method for integration of medical i...,1 . A system for integrating medical informati...,"['G06F17/30', 'G16H70/00']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60"


In [32]:
import ast


# Create a new column to store the common code
result_df['exact_match_code'] = ''

# Iterate through rows and compare 'sub_classes' and 'query_codes_G16H'
for index, row in result_df.iterrows():
    sub_classes_str = row['sub_classes']  # Data format in this field "['A61B5/00', 'G16H40/67']"
    query_codes_G16H = row['query_codes_G16H']  # Data format in this field 'G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H'
    
    # Convert the sub_classes string to a list
    sub_classes = ast.literal_eval(sub_classes_str)
    
    # Split the codes into lists
    sub_class_list = [code.strip() for code in sub_classes]
    query_codes_list = query_codes_G16H.split(',')
    
    # Check for common codes
    exact_match_code = [code for code in sub_class_list if code in query_codes_list]
    
    # Join the common codes into a single string
    exact_match_code_str = ','.join(exact_match_code)
    
    # Update the 'exact_match_code' column with the exact_match_code
    result_df.at[index, 'exact_match_code'] = exact_match_code_str
    
    # Debugging statements
    #print(f'Row {index}: sub_classes={sub_classes}, query_codes_G16H={query_codes_G16H}, common_codes={common_codes_str}')

# Display the updated DataFrame
result_df


,publication_number,title,first_claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code
0,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
1,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
2,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
3,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
4,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4895,US2018068076A1,Systems and methods for semantic search and ex...,1 . A system that facilitates using user-enter...,"['G16H50/20', 'G06F16/90344']","['G16H', 'G06F']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",
4896,US2007055543A1,Medical resource estimation and simulation system,1 . A healthcare decision support system for s...,"['G16H10/60', 'G16Z99/00']","['G16H', 'G16Z']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",G16H10/60
4897,US2007055545A1,System and user interface for processing patie...,1 . A system for use in processing patient cli...,"['G06F19/00', 'G16H50/20']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",
4898,US2006271556A1,System and method for integration of medical i...,1 . A system for integrating medical informati...,"['G06F17/30', 'G16H70/00']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",


In [33]:
# result_df.sample(n=100)

In [34]:
# Calculate the count of 'exact_match_code' for each group and assign it to all rows within the group
result_df['count_exact_match'] = result_df.groupby('query_publication_numbers')['exact_match_code'].transform(lambda x: x[x != ''].count())
result_df

,publication_number,title,first_claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code,count_exact_match
0,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
1,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
2,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
3,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
4,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4895,US2018068076A1,Systems and methods for semantic search and ex...,1 . A system that facilitates using user-enter...,"['G16H50/20', 'G06F16/90344']","['G16H', 'G06F']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",,16
4896,US2007055543A1,Medical resource estimation and simulation system,1 . A healthcare decision support system for s...,"['G16H10/60', 'G16Z99/00']","['G16H', 'G16Z']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",G16H10/60,16
4897,US2007055545A1,System and user interface for processing patie...,1 . A system for use in processing patient cli...,"['G06F19/00', 'G16H50/20']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",,16
4898,US2006271556A1,System and method for integration of medical i...,1 . A system for integrating medical informati...,"['G06F17/30', 'G16H70/00']","['G06F', 'G16H']",32,1.0,US20230138516A1,"G16H,G16H10/00,G06Q10/1093,G06Q10/10,G16,G06Q1...",1-27. (canceled) 28. A system to manage record...,[32],"G16H,G16H10/00,G16H40/20,G16H40/00,G16H10/60",,16


In [36]:
filtered_df = result_df[result_df['query_publication_numbers'] == 'US20230238130A1']
filtered_df

,publication_number,title,first_claim,sub_classes,sub_class,topics,prob,query_publication_numbers,query_class_codes,query_claim,query_predicted_topics,query_codes_G16H,exact_match_code,count_exact_match
0,US2008198033A1,Device for Communicating with a Voice-Disabled...,1 . A hand-held communication device comprisin...,"['G09B21/00', 'G16H40/63']","['G09B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
1,US2007118397A1,Monitoring of medical conditions,1 . A device for use in the provision of a med...,"['G16H10/20', 'A61B5/24']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
2,US2012303331A1,Adapter between scale and vital signs monitor,"1 . A method comprising: receiving, at an adap...","['G06F15/00', 'G16H40/63']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
3,US2012286959A1,Automated Allergy Alerts,1 . A wireless communication device comprising...,"['G16H10/60', 'G08B21/043']","['G16H', 'G08B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
4,US2012280049A1,Personal Health Record (PHR) ID card claiming ...,"1 . A customizable, portable identification an...","['G06K19/06', 'G16H10/65']","['G06K', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
5,US2012265556A1,Method and device for remote controlled applic...,55 . A device useful for monitoring of patient...,"['G16H40/67', 'A61B5/0022']","['G16H', 'A61B']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",G16H40/67,12
6,US2012239705A1,Mobile information system for 12-lead ecg,1 . A 12-leads electrocardiograph mobile infor...,"['G06F17/30', 'G16H10/60']","['G06F', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
7,US2010217803A1,Interface device for communication between a m...,1 . A wireless interface device for communicat...,"['G16H40/67', 'G16H40/67']","['G16H', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...","G16H40/67,G16H40/67",12
8,US2010253509A1,Personal environmental monitoring method and s...,"1 . A portable personal environmental monitor,...","['G08B1/08', 'G16H50/30']","['G08B', 'G16H']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
9,US2010283601A1,Medication usage monitoring and reminding devi...,"1 ) A device for monitoring medication usage, ...","['G16H10/60', 'A61J7/0418']","['G16H', 'A61J']",16,1.0,US20230238130A1,"A61B5/14551,A61,A61B5/0295,A61B5/6829,G,A61B5/...",1. A physiological monitoring device comprisin...,[16],"G16H40/00,G16H40/60,G16H40/67,G16H10/40,G16H,G...",,12
